In [1]:
import pandas as pd

In [9]:
df1=pd.read_csv('ir_avg.csv')
print(df1)

            Date  Average
0    Feb-07-2025    6.375
1    Dec-06-2024    6.500
2    Oct-09-2024    6.500
3    Aug-08-2024    6.500
4    Jun-07-2024    6.500
..           ...      ...
229  May-12-2000    9.500
230  Apr-23-2000   11.000
231  Mar-09-2000   12.000
232  Feb-19-2000    9.600
233  Jan-30-2000    8.000

[234 rows x 2 columns]


In [11]:
# Convert the "Date" column to datetime format
df1["Date"] = pd.to_datetime(df1["Date"], format="%b-%d-%Y")
# df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

# Display the updated dataframe
df1.head()


,Date,Average
0,2025-02-07,6.375
1,2024-12-06,6.500
2,2024-10-09,6.500
3,2024-08-08,6.500
4,2024-06-07,6.500


In [12]:
df1

,Date,Average
0,2025-02-07,6.375
1,2024-12-06,6.500
2,2024-10-09,6.500
3,2024-08-08,6.500
4,2024-06-07,6.500
...,...,...
229,2000-05-12,9.500
230,2000-04-23,11.000
231,2000-03-09,12.000
232,2000-02-19,9.600


In [14]:
df1.to_csv('ir_avg_date.csv',index=False)
df1

,Date,Average
0,2025-02-07,6.375
1,2024-12-06,6.500
2,2024-10-09,6.500
3,2024-08-08,6.500
4,2024-06-07,6.500
...,...,...
229,2000-05-12,9.500
230,2000-04-23,11.000
231,2000-03-09,12.000
232,2000-02-19,9.600


In [20]:
import numpy as np

# Initialize a new DataFrame to store expanded data
expanded_df = pd.DataFrame(columns=df1.columns)

# Define a function to generate random dates within a range
def random_dates(base_date, num_dates=10, days_range=30):
    return [base_date + pd.Timedelta(days=np.random.randint(1, days_range)) for _ in range(num_dates)]

# Generate additional rows for each original row
for _, row in df1.iterrows():
    # Generate 10 new rows with random values around the original row
    new_rows = pd.DataFrame({
        "Date": random_dates(row["Date"], 20),
        "Average": np.random.uniform(row["Average"] - 0.1, row["Average"] + 0.1, 20).round(3)
    })
    
# Append the original row and generated rows
expanded_df = pd.concat([expanded_df, pd.DataFrame([row]), new_rows], ignore_index=True)

# Sort by date
expanded_df = expanded_df.sort_values(by="Date").reset_index(drop=True)

# Display the expanded DataFrame
expanded_df  # Show first 20 rows for verification


C:\Users\Aayush Kuthe\AppData\Local\Temp\ipykernel_21240\1179022130.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  expanded_df = pd.concat([expanded_df, pd.DataFrame([row]), new_rows], ignore_index=True)


,Date,Average
0,2000-01-30,8.000
1,2000-02-02,8.026
2,2000-02-04,8.099
3,2000-02-05,7.956
4,2000-02-07,7.975
5,2000-02-09,7.912
6,2000-02-11,8.072
7,2000-02-11,7.944
8,2000-02-12,8.092
9,2000-02-13,7.923


In [21]:
# df1=pd.read_csv('expanded_ir_avg.csv')
expanded_df

,Date,Average
0,2000-01-30,8.000
1,2000-02-02,8.026
2,2000-02-04,8.099
3,2000-02-05,7.956
4,2000-02-07,7.975
5,2000-02-09,7.912
6,2000-02-11,8.072
7,2000-02-11,7.944
8,2000-02-12,8.092
9,2000-02-13,7.923


In [11]:
import pandas as pd
import numpy as np

# Load the original CSV file
file_path = "loan.csv"  # Change to your actual file path
df1 = pd.read_csv(file_path)

# Convert "Date" column to datetime format
df1["Date"] = pd.to_datetime(df1["Date"], errors="coerce").dt.date

# Function to interpolate average values in sorted order
def interpolate_averages(df):
    interpolated_data = []
    interpolated_dates = []
    for i in range(len(df) - 1):
        date1, date2 = df.loc[i, "Date"], df.loc[i + 1, "Date"]
        avg1, avg2 = df.loc[i, "Loan Rate"], df.loc[i + 1, "Loan Rate"]
        
        # Generate dates between date1 and date2
        date_range = pd.date_range(start=date1, end=date2, periods=10)[1:-1].date  # Exclude endpoints
        
        # If both values are the same, repeat the same value for interpolated entries
        if avg1 == avg2:
            new_averages = np.full(len(date_range), avg1)
        elif avg1 < avg2:
            new_averages = np.linspace(avg1, avg2, num=len(date_range))  # Increasing order
        else:
            new_averages = np.linspace(avg1, avg2, num=len(date_range))[::-1]  # Decreasing order
        
        # Round off to 3 decimal places
        new_averages = np.round(new_averages, 3)
        
        # Append interpolated values
        interpolated_dates.extend(date_range)
        interpolated_data.extend(new_averages)
    
    return interpolated_dates, interpolated_data

# Generate interpolated dates and average values
interpolated_dates, interpolated_averages = interpolate_averages(df1)

# Create a new DataFrame with interpolated values
new_data = pd.DataFrame({
    "Date": interpolated_dates,
    "Loan Rate": interpolated_averages
})

# Combine the old and new data
df1_extended = pd.concat([df1, new_data], ignore_index=True)

# Convert "Date" column to datetime again to ensure consistency
df1_extended["Date"] = pd.to_datetime(df1_extended["Date"]).dt.date

# Sort data by date
df1_extended = df1_extended.sort_values(by="Date").reset_index(drop=True)

# Save the extended dataset
output_file_path = "extended_loan_2k.csv"  # Change to your desired save path
df1_extended.to_csv(output_file_path, index=False)

print(f"Extended dataset saved as: {output_file_path}")


Extended dataset saved as: extended_loan_2k.csv


In [ ]:
import pandas as pd
import numpy as np

# Load the original CSV file
file_path = "loan.csv"  # Update with the correct file path
df = pd.read_csv(file_path)

# Convert "Date" column to datetime format
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")

# Sort data by Date to ensure chronological order
df = df.sort_values(by="Date").reset_index(drop=True)

# Function to interpolate missing GDP values while ensuring sorting logic
def interpolate_gdp(df):
    for i in range(len(df) - 1):
        if pd.isna(df.loc[i, "Loan Rate"]) and not pd.isna(df.loc[i + 1, "Loan Rate"]):
            lower_limit = df.loc[i - 1, "Loan Rate"] if i > 0 else df.loc[i + 1, "Loan Rate"]
            upper_limit = df.loc[i + 1, "Loan Rate"]
            
            # Generate interpolated values
            df.loc[i, "Loan Rate"] = round((lower_limit + upper_limit) / 2, 3)
    
    # Use linear interpolation for remaining missing values
df["Loan Rate"] = df["Loan Rate"].interpolate(method="linear").round(3)

# Apply interpolation function
interpolate_gdp(df)

# Save the processed dataset
output_file_path = "loan_interpolated.csv"
df.to_csv(output_file_path, index=False)

print(f"Interpolated dataset saved as: {output_file_path}")


Interpolated dataset saved as: loan_interpolated.csv


In [1]:
import pandas as pd
import numpy as np

# Load the CSV file
file_path = "savings.csv"  # Update with the correct file path
df = pd.read_csv(file_path)

# Convert "Date" column to datetime format
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")

# Sort data by Date to ensure chronological order
df = df.sort_values(by="Date").reset_index(drop=True)

# Function to interpolate missing values while ensuring sorting logic
def interpolate_savings(df, columns):
    for col in columns:
        for i in range(len(df) - 1):
            if pd.isna(df.loc[i, col]) and not pd.isna(df.loc[i + 1, col]):
                lower_limit = df.loc[i - 1, col] if i > 0 else df.loc[i + 1, col]
                upper_limit = df.loc[i + 1, col]
                
                # Generate interpolated values
                df.loc[i, col] = round((lower_limit + upper_limit) / 2, 3)
    
    # Use linear interpolation for remaining missing values
    df[columns] = df[columns].interpolate(method="linear").round(3)

# Apply interpolation function to relevant columns
interpolate_savings(df, ["1 Year", "2 Year", "3 Year", "5 Year"])

# Save the processed dataset
output_file_path = "savings_interpolated.csv"
df.to_csv(output_file_path, index=False)

print(f"Interpolated dataset saved as: {output_file_path}")


Interpolated dataset saved as: savings_interpolated.csv
